In [8]:
from ftplib import FTP_TLS
from pathlib import Path 
import json
import os
import pandas as pd

In [9]:
from datetime import datetime


def DownFile(FechaInicial: str)->str:
    # Connect to the FTP server (replace with your actual details)
    ftps  = FTP_TLS()
    ftps .connect('xmftps.xm.com.co', 210)  # default port is 210

    # Secure the control connection
    ftps .auth()
    ftps .prot_p()  # Switch to secure data connection (important!)

    ftps .login(r'1060588666', 'Alejo230706*+')

    # Obtener mes y día de la fecha inicial
    # Transformar string en fecha
    fecha = datetime.strptime(FechaInicial, "%d/%m/%Y")

    # Obtener mes y día
    year= fecha.year
    mes = fecha.month
    dia = fecha.day

    # Navigate to the directory you want to access
    ftps.cwd(f"/INFORMACION_XM/Publico/DESPACHO/{year:04d}-{mes:02d}")

    # List files
    files = ftps.nlst()
    # print("Available files:", files)



    # Download condiciones iniciales de planta
    
    pathfile=f"C:\Información XM\Publico\Despacho\{year:04d}-{mes:02d}\\"
    
    filename = f"dCondIniP{mes:02d}{dia:02d}.TXT"
    
    print(pathfile + filename)
    with open(pathfile + filename, 'wb') as f:
        ftps.retrbinary(f"RETR {filename}", f.write)

    # Download condiciones iniciales de unidad
    filename = f"dCondIniU{mes:02d}{dia:02d}.TXT"

    print(pathfile + filename)
    with open(pathfile + filename, 'wb') as f:
        ftps.retrbinary(f"RETR {filename}", f.write)

    ftps.quit()
    # print(f"{filename} downloaded successfully.")

    return "Finalizado"

In [ ]:
def CreateFile(FechaInicial,sPathfile)->str:

    #Leer archivo de mapeos
    spathFileMap=r'C:\Alejo\cops\Data\Mapeos.xlsx'
    mapping_df=pd.read_excel(spathFileMap,sheet_name='NomRecursos')

    fecha = datetime.strptime(FechaInicial, "%d/%m/%Y")

    # Obtener mes y día
    year= fecha.year
    mes = fecha.month
    dia = fecha.day

    # Download condiciones iniciales de planta
    
    pathfile=f"C:\Información XM\Publico\Despacho\{year:04d}-{mes:02d}\\"

    filename = f"dCondIniP{mes:02d}{dia:02d}.TXT"
    # print(filename)
    pathfile = pathfile + filename

    # Load the file into a dataframe
    df = pd.read_csv(pathfile, delimiter=',',header=0,encoding="ISO-8859-1")
    # df['ConfMod']= df['Conf_Pini-1'].str.split('_').str[0].fillna(0).astype(int)
    # Remove leading and trailing spaces from column names
    df.columns = df.columns.str.strip()

    # Eliminar filas donde la columna 'Recurso' coincide con varios strings específicos
    strings_to_remove = ['MIEL I', 'SOGAMOSO']
    df = df[~df['Planta'].isin(strings_to_remove)]
    df = df.sort_values(by='Planta')
    l_cols=list(df.columns)
    l_cols.append('RecCops')
    df=df.merge(mapping_df,left_on=['Planta'],right_on=['RecDespacho'],how='left')[l_cols]
    # Save the dataframe to a JSON file
    sFile=r"CIPlanta.txt"
    output_path = os.path.join(sPathfile, sFile)
    df.to_csv(output_path, index=False)

    mapping_df=pd.read_excel(spathFileMap,sheet_name='NomUnidad')

    pathfile=f"C:\Información XM\Publico\Despacho\{year:04d}-{mes:02d}\\"

    filename = f"dCondIniU{mes:02d}{dia:02d}.TXT"
    # print(filename)
    pathfile = pathfile + filename

    # Load the file into a dataframe
    df = pd.read_csv(pathfile, delimiter=',',header=0,encoding="ISO-8859-1")

    # Remove leading and trailing spaces from column names
    df.columns = df.columns.str.strip()

    df = df.sort_values(by='Unidad')
    l_cols=list(df.columns)
    l_cols.append('UniCops')
    df=df.merge(mapping_df,left_on=['Unidad'],right_on=['UniDespacho'],how='left')[l_cols]

    # Save the dataframe to a JSON file
    sFile=r"CIUnidad.txt"
    output_path = os.path.join(sPathfile, sFile)
    df.to_csv(output_path, index=False)

    

    return df

In [11]:

# Ruta del archivo
# script_dir = Path(__file__).resolve()
# script_dir=script_dir.parent.parent.parent
# sPathfile=os.path.join(script_dir,r"Modules\Utils\ArchivosAux",sFile)
sPathFolder=r"C:\Alejo\cops\Modules\Utils\ArchivosAux"

sFile=r"Parametros.json"
sPathfile=os.path.join(sPathFolder,sFile)

# Open and load the JSON file
with open(sPathfile,'r') as f:
    data = json.load(f)

# Almancenar los parámetros en variables python
year=data['Ano']
mes=data['Mes']
Carpeta=data['Carpeta']
FechaInicial=data['Fecha_Inicial']
FechaFinal=data['Fecha_Final']

msj=DownFile(FechaInicial)
df=CreateFile(FechaInicial,sPathFolder)
df

C:\Información XM\Publico\Despacho\2025-05\dCondIniP0513.TXT
C:\Información XM\Publico\Despacho\2025-05\dCondIniU0513.TXT


,Unidad,COMBUSTIBLE,DISPPINI1,GP,NARRANQUESPINI1,PRUEBAS,TAPUBLICAR,TDISPPINI1,TFL,TL,UniCops
0,BARRANQUILLA 3,Gas,60.0,0.0,0,0,10,4296,1161,0,BARRANQUILLA_3
1,BARRANQUILLA 4,Gas,60.0,0.0,0,0,10,4536,110,0,BARRANQUILLA_4
2,CARTAGENA 1,CombustÃ³leo,30.0,0.0,0,0,10,456,3986,0,CARTAGENA_1
3,CARTAGENA 2,CombustÃ³leo,39.0,0.0,0,0,10,384,481,0,CARTAGENA_2
4,CARTAGENA 3,CombustÃ³leo,0.0,0.0,0,0,10,0,5602,0,CARTAGENA_3
...,...,...,...,...,...,...,...,...,...,...,...
70,TESORITO 9,Gas,18.0,0.0,0,0,10,2640,2282,0,TESORITO_9
71,ZIPAEMG 2,CarbÃ³n,36.0,0.0,0,0,10,24,305,0,ZIPAEMG_3
72,ZIPAEMG 3,CarbÃ³n,0.0,0.0,0,0,10,0,1623,0,ZIPAEMG_4
73,ZIPAEMG 4,CarbÃ³n,64.0,0.0,0,0,10,2166,2057,0,ZIPAEMG_5
